# Beer (Nature Communications) Expert Panel Only — Tabular Synthetic Generation & Manifold Preservation (4090 eGPU)
이 노트북은 Zenodo 레코드 **10.5281/zenodo.10653704**에 포함된 파일들 중,
- **화학 성분(chemistry) + 전문가 패널(Trained Panel) 관능평가** 데이터를 사용하고
- **RateBeer(일반인 리뷰) 관련 데이터/모델은 사용하지 않으며**
- **High-dimensional small data** 환경에서 **True Manifold 보존 합성**을 목표로  
  **Statistical vs Generative AI (VAE/CTGAN)**를 비교합니다.

---

## 데이터 출처 구조(요약)
저자 노트북(“Jupyter Notebook for training machine learning models.ipynb”) 기준으로:
- 화학 데이터: `Supplemental File S1`  
- 전문가 패널 관능 데이터: `Supplemental File S4`  
- 두 데이터를 `beer`(index) 기준으로 join하여,
  - **X (features)**: `acetaldehyde` ~ `sulfur_sum`
  - **Y (targets)**: `A_malt_all` ~ `overall`
  - stratify 라벨로 `tasting_category_fine` 사용  
  를 수행합니다.

> 이 노트북은 위 구조를 **CSV로 변환하지 않아도** Excel(.xlsx)에서 직접 로드하도록 구현했습니다.

## 0. (필수) 로컬 데이터 폴더 경로 설정
사용자 PC 기준 경로:
`C:\Users\eys63\Desktop\맥주데이터실험\data`

아래 셀에서 `DATA_DIR`만 맞으면 나머지는 자동 탐지합니다.

In [1]:

from pathlib import Path

DATA_DIR = Path(r"C:\Users\eys63\Desktop\맥주데이터실험\data")
XLSX_PATH = DATA_DIR / "Supplemental Files and Figure source files.xlsx"

print("DATA_DIR exists:", DATA_DIR.exists(), "|", DATA_DIR)
print("XLSX exists:", XLSX_PATH.exists(), "|", XLSX_PATH)

# 폴더 내 파일 확인
for p in sorted(DATA_DIR.glob("*")):
    print(p.name)

DATA_DIR exists: True | C:\Users\eys63\Desktop\맥주데이터실험\data
XLSX exists: True | C:\Users\eys63\Desktop\맥주데이터실험\data\Supplemental Files and Figure source files.xlsx
init_modeling.py
Jupyter Notebook for training machine learning models.ipynb
Machine learning models transcript.py
Main Figure Generator.Rmd
Partial Dependence Plots with SHAP.ipynb
RateBeer_Appreciation_Gradient_Boost_Regressor.pkl
RateBeer_Full_Model_Gradient_Boost_Regressor.pkl
README.txt
Sensory data quality control - ANOVA on repeated samples.Rmd
SHAP analysis.ipynb
Supplemental Figure generator.Rmd
Supplemental Files and Figure source files.xlsx


## 1. 라이브러리 Import + 4090 eGPU 최적화 설정
- VAE/MLP: PyTorch + CUDA + AMP(BF16/FP16)
- CTGAN: 설치되어 있으면 GPU(cuda=True) 사용 시도

In [2]:

import os, math, time, random
import numpy as np
import pandas as pd

from scipy import stats
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

USE_AMP = (DEVICE.type == "cuda")
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
SCALER = torch.cuda.amp.GradScaler(enabled=(USE_AMP and AMP_DTYPE == torch.float16))
print("USE_AMP =", USE_AMP, "| AMP_DTYPE =", AMP_DTYPE, "| GradScaler enabled =", SCALER.is_enabled())

DEVICE = cuda
USE_AMP = True | AMP_DTYPE = torch.bfloat16 | GradScaler enabled = False


C:\Users\eys63\AppData\Local\Temp\ipykernel_17284\1360775470.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler(enabled=(USE_AMP and AMP_DTYPE == torch.float16))


## 2. Excel(.xlsx)에서 S1(chem) / S4(expert panel) 시트 자동 탐지 & 로드

저자 노트북에서는 CSV로 저장한 뒤 `Supplemental File S1.csv`, `Supplemental File S4.csv`를 읽습니다.  
하지만 사용자는 원본 Excel 파일을 갖고 있으므로, 이 노트북은:

1) **시트 목록 출력**
2) 아래 키 컬럼을 포함하는 시트를 자동 탐지  
   - chem: `acetaldehyde`, `sulfur_sum`  
   - expert panel: `A_malt_all`, `overall`
3) `beer` 컬럼이 있으면 index로 설정
4) inner join

> 만약 컬럼명이 조금 다르면, 자동 탐지가 실패할 수 있습니다. 그 경우 셀 출력(시트 목록/컬럼)을 보고 직접 시트명을 지정하면 됩니다.

In [3]:

def list_excel_sheets(xlsx_path: Path):
    xls = pd.ExcelFile(xlsx_path)
    return xls, xls.sheet_names

def find_sheet_by_required_columns(xls: pd.ExcelFile, required_cols: list[str]) -> str | None:
    for sh in xls.sheet_names:
        # header만 읽기 (빠름)
        try:
            df0 = xls.parse(sh, nrows=0)
        except Exception:
            continue
        cols = [c.strip() if isinstance(c, str) else c for c in df0.columns]
        if all(rc in cols for rc in required_cols):
            return sh
    return None

xls, sheets = list_excel_sheets(XLSX_PATH)
print("Excel sheets:", len(sheets))
print(sheets[:30], "..." if len(sheets) > 30 else "")

chem_sheet = find_sheet_by_required_columns(xls, ["acetaldehyde", "sulfur_sum"])
panel_sheet = find_sheet_by_required_columns(xls, ["A_malt_all", "overall"])

print("Auto-detected chem_sheet :", chem_sheet)
print("Auto-detected panel_sheet:", panel_sheet)

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [ ]:

# --- if auto-detect fails, you can manually set here ---
# chem_sheet = "Supplemental File S1"
# panel_sheet = "Supplemental File S4"

chem_df = xls.parse(chem_sheet)
panel_df = xls.parse(panel_sheet)

print("chem_df shape:", chem_df.shape)
print("panel_df shape:", panel_df.shape)
display(chem_df.head(3))
display(panel_df.head(3))

## 3. (Optional) 저자 노트북 호환을 위해 CSV로 Export
저자 노트북/스크립트가 `Supplemental File S1.csv`, `Supplemental File S4.csv`를 기대하므로,
원하면 아래 셀로 동일 파일명을 생성할 수 있습니다.

In [ ]:

EXPORT_CSV_FOR_AUTHORS = True
if EXPORT_CSV_FOR_AUTHORS:
    out_s1 = DATA_DIR / "Supplemental File S1.csv"
    out_s4 = DATA_DIR / "Supplemental File S4.csv"
    chem_df.to_csv(out_s1, index=False, encoding="utf-8-sig")
    panel_df.to_csv(out_s4, index=False, encoding="utf-8-sig")
    print("Exported:", out_s1, out_s4)
else:
    print("Skip export")

## 4. Expert Panel Only Modeling Dataset 생성 (X,Y,Stratify label)

저자 노트북 구조를 그대로 따릅니다:
- index: `beer` (존재 시)
- join: chem + panel
- X: `acetaldehyde` ~ `sulfur_sum`
- Y: `A_malt_all` ~ `overall`
- stratify: `tasting_category_fine` (존재 시)

추가:
- numeric만 유지
- 결측치(mean impute)

In [ ]:

def set_index_if_exists(df: pd.DataFrame, col: str) -> pd.DataFrame:
    if col in df.columns:
        return df.set_index(col)
    return df

chem_df_i = set_index_if_exists(chem_df, "beer")
panel_df_i = set_index_if_exists(panel_df, "beer")

# join on index (beer)
df = chem_df_i.join(panel_df_i, how="inner", lsuffix="_chem", rsuffix="_panel")
print("Joined df shape:", df.shape)

# stratify label (optional)
strat_col = "tasting_category_fine"
y_class = df[strat_col] if strat_col in df.columns else None
print("Has stratify column:", strat_col in df.columns)

# feature range & target range (authors' selection)
def slice_range_columns(df: pd.DataFrame, start: str, end: str) -> list[str]:
    cols = list(df.columns)
    if start in cols and end in cols:
        i0, i1 = cols.index(start), cols.index(end)
        if i0 <= i1:
            return cols[i0:i1+1]
        else:
            return cols[i1:i0+1]
    return []

x_cols = slice_range_columns(df, "acetaldehyde", "sulfur_sum")
y_cols = slice_range_columns(df, "A_malt_all", "overall")

print("X cols detected:", len(x_cols))
print("Y cols detected:", len(y_cols))
print("Example X cols:", x_cols[:5], "...", x_cols[-5:] if len(x_cols) >= 5 else "")
print("Example Y cols:", y_cols[:5], "...", y_cols[-5:] if len(y_cols) >= 5 else "")

# fallback if range slicing failed
if len(x_cols) == 0:
    # take numeric columns from chem part: prefer columns that exist in chem_df_i
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # remove y range if exists
    x_cols = [c for c in numeric_cols if c not in y_cols]
    print("[Fallback] X cols =", len(x_cols))

if len(y_cols) == 0:
    # take numeric columns from panel part: prefer columns that exist in panel_df_i
    panel_numeric = panel_df_i.select_dtypes(include=[np.number]).columns.tolist()
    # keep those present in joined df
    y_cols = [c for c in panel_numeric if c in df.columns]
    print("[Fallback] Y cols =", len(y_cols))

X_df = df[x_cols].apply(pd.to_numeric, errors="coerce")
Y_df = df[y_cols].apply(pd.to_numeric, errors="coerce")

print("X_df:", X_df.shape, "| Y_df:", Y_df.shape)
display(X_df.describe().T.head(5))
display(Y_df.describe().T.head(5))

## 5. 전처리 (Mean Impute + Log1p + MinMax)
논문 요구 사항 반영:
- 다수 값이 0 근처 + long-tail → **log1p**
- 생성/학습 안정성 → **Min-Max scaling**

주의:
- 혹시 feature에 음수가 있다면 log1p가 불가능하므로, **컬럼별 shift**를 자동 적용합니다.

In [ ]:

class BeerPreprocessor:
    def __init__(self):
        self.imputer_x = SimpleImputer(strategy="mean")
        self.imputer_y = SimpleImputer(strategy="mean")
        self.x_scaler = MinMaxScaler()
        self.y_scaler = MinMaxScaler()
        self.shift_ = None
        self.fitted_ = False

    def fit(self, X: np.ndarray, y: np.ndarray):
        X_imp = self.imputer_x.fit_transform(X)
        y_imp = self.imputer_y.fit_transform(y)

        # per-feature shift to make non-negative for log1p
        min_per_col = np.nanmin(X_imp, axis=0)
        shift = np.where(min_per_col < 0, -min_per_col, 0.0).astype(np.float32)
        self.shift_ = shift

        X_shift = X_imp + shift
        X_log = np.log1p(np.clip(X_shift, 0, None))

        self.x_scaler.fit(X_log)
        self.y_scaler.fit(y_imp)

        self.fitted_ = True
        return self

    def transform_X(self, X: np.ndarray) -> np.ndarray:
        assert self.fitted_
        X_imp = self.imputer_x.transform(X)
        X_shift = X_imp + self.shift_
        X_log = np.log1p(np.clip(X_shift, 0, None))
        Xs = self.x_scaler.transform(X_log)
        return Xs.astype(np.float32)

    def transform_y(self, y: np.ndarray) -> np.ndarray:
        assert self.fitted_
        y_imp = self.imputer_y.transform(y)
        ys = self.y_scaler.transform(y_imp)
        return ys.astype(np.float32)

    def inverse_transform_X(self, Xs: np.ndarray) -> np.ndarray:
        assert self.fitted_
        X_log = self.x_scaler.inverse_transform(Xs)
        X_shift = np.expm1(X_log)
        X = X_shift - self.shift_
        return X.astype(np.float32)

    def inverse_transform_y(self, ys: np.ndarray) -> np.ndarray:
        assert self.fitted_
        y = self.y_scaler.inverse_transform(ys)
        return y.astype(np.float32)

In [ ]:

X = X_df.values.astype(np.float32)
y = Y_df.values.astype(np.float32)

# train/test split (stratify if available)
X_train, X_test, y_train, y_test, yclass_train, yclass_test = train_test_split(
    X, y, y_class.values if y_class is not None else np.zeros(len(X)),
    test_size=0.2, random_state=42,
    stratify=(y_class.values if y_class is not None else None)
)

pre = BeerPreprocessor().fit(X_train, y_train)
X_train_s = pre.transform_X(X_train)
X_test_s  = pre.transform_X(X_test)
y_train_s = pre.transform_y(y_train)
y_test_s  = pre.transform_y(y_test)

print("Train:", X_train_s.shape, y_train_s.shape, "| Test:", X_test_s.shape, y_test_s.shape)

## 6. 합성 모델 (Baselines + VAE + CTGAN)
- Baselines: Bootstrapping / Independent KDE / Gaussian Copula
- Proposed: VAE (직접 구현, GPU/AMP), CTGAN (설치 시)

> CTGAN이 설치되어 있지 않으면 자동 skip합니다.

In [ ]:

class BaseSynthesizer:
    name = "BaseSynthesizer"
    def fit(self, Xs: np.ndarray, ys: np.ndarray):
        raise NotImplementedError
    def sample(self, n: int):
        raise NotImplementedError

class BootstrapSynthesizer(BaseSynthesizer):
    name = "Bootstrapping"
    def __init__(self, random_state: int = 0):
        self.rng = np.random.default_rng(random_state)
    def fit(self, Xs, ys):
        self.Xs = Xs
        self.ys = ys
        return self
    def sample(self, n):
        idx = self.rng.integers(0, len(self.Xs), size=n)
        return self.Xs[idx], self.ys[idx]

In [ ]:

from sklearn.neighbors import KernelDensity

class IndependentKDESynthesizer(BaseSynthesizer):
    name = "IndependentKDE"
    def __init__(self, bandwidth: float = 0.05, kernel: str = "gaussian", random_state: int = 0):
        self.bandwidth = bandwidth
        self.kernel = kernel
        self.rng = np.random.default_rng(random_state)

    def fit(self, Xs: np.ndarray, ys: np.ndarray):
        self.d_x = Xs.shape[1]
        self.d_y = ys.shape[1]
        Z = np.concatenate([Xs, ys], axis=1)
        self.models = []
        self.is_constant = []
        self.const_value = []

        for j in range(Z.shape[1]):
            col = Z[:, [j]]
            if np.allclose(col, col[0]):
                self.models.append(None)
                self.is_constant.append(True)
                self.const_value.append(float(col[0, 0]))
            else:
                kde = KernelDensity(kernel=self.kernel, bandwidth=self.bandwidth)
                kde.fit(col)
                self.models.append(kde)
                self.is_constant.append(False)
                self.const_value.append(0.0)
        return self

    def sample(self, n: int):
        d_total = self.d_x + self.d_y
        Zs = np.zeros((n, d_total), dtype=np.float32)
        for j in range(d_total):
            if self.is_constant[j]:
                Zs[:, j] = self.const_value[j]
            else:
                samp = self.models[j].sample(n_samples=n, random_state=int(self.rng.integers(0, 1_000_000)))
                Zs[:, j] = samp[:, 0]
        Zs = np.nan_to_num(Zs, nan=0.0, posinf=1.0, neginf=0.0)
        Zs = np.clip(Zs, 0.0, 1.0)
        return Zs[:, :self.d_x], Zs[:, self.d_x:]

In [ ]:

def _rank_to_u(x: np.ndarray) -> np.ndarray:
    n = len(x)
    ranks = stats.rankdata(x, method="average")
    u = (ranks - 0.5) / n
    return np.clip(u, 1e-6, 1 - 1e-6)

def _make_psd_corr(C: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    w, V = np.linalg.eigh(C)
    w = np.clip(w, eps, None)
    Cp = (V * w) @ V.T
    d = np.sqrt(np.clip(np.diag(Cp), eps, None))
    Cp = Cp / np.outer(d, d)
    Cp[np.diag_indices_from(Cp)] = 1.0
    return Cp

class GaussianCopulaSynthesizer(BaseSynthesizer):
    name = "GaussianCopula"
    def __init__(self, random_state: int = 0):
        self.rng = np.random.default_rng(random_state)

    def fit(self, Xs: np.ndarray, ys: np.ndarray):
        self.d_x = Xs.shape[1]
        self.d_y = ys.shape[1]
        Z = np.concatenate([Xs, ys], axis=1).astype(np.float64)

        self.is_constant = np.zeros(Z.shape[1], dtype=bool)
        self.const_value = np.zeros(Z.shape[1], dtype=np.float64)

        U = np.zeros_like(Z)
        for j in range(Z.shape[1]):
            col = Z[:, j]
            if np.allclose(col, col[0]):
                self.is_constant[j] = True
                self.const_value[j] = col[0]
                U[:, j] = 0.5
            else:
                U[:, j] = _rank_to_u(col)

        G = norm.ppf(U)

        lw = LedoitWolf().fit(G)
        cov = lw.covariance_
        d = np.sqrt(np.clip(np.diag(cov), 1e-12, None))
        corr = cov / np.outer(d, d)
        corr[np.diag_indices_from(corr)] = 1.0
        self.corr = _make_psd_corr(corr)
        self.Z_ref = Z
        return self

    def sample(self, n: int):
        d_total = self.d_x + self.d_y
        Gs = self.rng.multivariate_normal(mean=np.zeros(d_total), cov=self.corr, size=n)
        U = norm.cdf(Gs)

        Zs = np.zeros((n, d_total), dtype=np.float64)
        for j in range(d_total):
            if self.is_constant[j]:
                Zs[:, j] = self.const_value[j]
            else:
                Zs[:, j] = np.quantile(self.Z_ref[:, j], U[:, j], method="linear")

        Zs = np.nan_to_num(Zs, nan=0.0, posinf=1.0, neginf=0.0)
        Zs = np.clip(Zs, 0.0, 1.0).astype(np.float32)
        return Zs[:, :self.d_x], Zs[:, self.d_x:]

In [ ]:

class TabularVAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 24, hidden=(256, 128), dropout: float = 0.05):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        enc = []
        d = input_dim
        for h in hidden:
            enc += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.encoder = nn.Sequential(*enc)
        self.fc_mu = nn.Linear(d, latent_dim)
        self.fc_logvar = nn.Linear(d, latent_dim)

        dec = []
        d = latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        dec += [nn.Linear(d, input_dim)]
        self.decoder = nn.Sequential(*dec)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return torch.sigmoid(self.decoder(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

class VAESynthesizer(BaseSynthesizer):
    name = "VAE"
    def __init__(
        self,
        latent_dim=24,
        hidden=(256, 128),
        beta=0.6,
        lr=2e-3,
        weight_decay=1e-4,
        max_epochs=600,
        batch_size=256,
        patience=50,
        noise_std=0.01,
        random_state=0,
        device=DEVICE,
    ):
        self.latent_dim = latent_dim
        self.hidden = hidden
        self.beta = beta
        self.lr = lr
        self.weight_decay = weight_decay
        self.max_epochs = max_epochs
        self.batch_size = batch_size
        self.patience = patience
        self.noise_std = noise_std
        self.random_state = random_state
        self.device = device

    def fit(self, Xs: np.ndarray, ys: np.ndarray):
        seed_everything(self.random_state)
        self.d_x = Xs.shape[1]
        self.d_y = ys.shape[1]
        Z = np.concatenate([Xs, ys], axis=1).astype(np.float32)
        Zt = torch.from_numpy(Z).to(self.device)

        n = Zt.shape[0]
        idx = torch.randperm(n, device=self.device)
        n_val = max(1, int(0.2 * n))
        val_idx = idx[:n_val]
        tr_idx  = idx[n_val:]
        Ztr, Zva = Zt[tr_idx], Zt[val_idx]

        self.model = TabularVAE(Z.shape[1], latent_dim=self.latent_dim, hidden=self.hidden).to(self.device)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        def loss_fn(x, x_hat, mu, logvar):
            recon = F.mse_loss(x_hat, x, reduction="mean")
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            return recon + self.beta * kld, recon.detach(), kld.detach()

        best = float("inf")
        best_state = None
        bad = 0
        bs = min(self.batch_size, len(Ztr))

        for epoch in range(1, self.max_epochs + 1):
            self.model.train()
            perm = torch.randperm(len(Ztr), device=self.device)
            tr_loss = 0.0

            for i in range(0, len(Ztr), bs):
                batch = Ztr[perm[i:i+bs]]
                if self.noise_std > 0:
                    batch_in = torch.clamp(batch + self.noise_std * torch.randn_like(batch), 0.0, 1.0)
                else:
                    batch_in = batch

                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                    x_hat, mu, logvar = self.model(batch_in)
                    loss, _, _ = loss_fn(batch, x_hat, mu, logvar)

                if SCALER.is_enabled():
                    SCALER.scale(loss).backward()
                    SCALER.step(opt)
                    SCALER.update()
                else:
                    loss.backward()
                    opt.step()

                tr_loss += float(loss.detach()) * len(batch)

            tr_loss /= len(Ztr)

            self.model.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                    x_hat, mu, logvar = self.model(Zva)
                    va_loss, _, _ = loss_fn(Zva, x_hat, mu, logvar)
                va_loss = float(va_loss.detach())

            if va_loss < best - 1e-6:
                best = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                bad = 0
            else:
                bad += 1

            if epoch % 50 == 0 or epoch == 1:
                print(f"[VAE] epoch {epoch:4d} | train {tr_loss:.6f} | val {va_loss:.6f} | best {best:.6f}")

            if bad >= self.patience:
                print(f"[VAE] Early stopping at epoch {epoch} (best val {best:.6f})")
                break

        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self

    def sample(self, n: int):
        self.model.eval()
        with torch.no_grad():
            z = torch.randn((n, self.latent_dim), device=self.device)
            with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                out = self.model.decode(z)
        Zs = out.float().clamp(0, 1).cpu().numpy()
        return Zs[:, :self.d_x], Zs[:, self.d_x:]

In [ ]:

class CTGANSynthesizerWrapper(BaseSynthesizer):
    name = "CTGAN"
    def __init__(self, epochs: int = 300, batch_size: int = 500, random_state: int = 0, use_cuda: bool = True):
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.use_cuda = use_cuda

    def fit(self, Xs: np.ndarray, ys: np.ndarray):
        self.d_x = Xs.shape[1]
        self.d_y = ys.shape[1]
        Z = np.concatenate([Xs, ys], axis=1)
        self.cols = [f"col_{i:03d}" for i in range(Z.shape[1])]
        df = pd.DataFrame(Z, columns=self.cols)

        try:
            from ctgan import CTGAN
            self.backend = "ctgan"
            self.model = CTGAN(
                epochs=self.epochs,
                batch_size=self.batch_size,
                verbose=True,
                cuda=(self.use_cuda and torch.cuda.is_available()),
            )
            self.model.fit(df, discrete_columns=[])
            return self
        except Exception as e_ctgan:
            print("[CTGAN] ctgan backend failed:", e_ctgan)

        # fallback sdv
        try:
            from sdv.single_table import CTGANSynthesizer
            from sdv.metadata import SingleTableMetadata
            self.backend = "sdv"
            meta = SingleTableMetadata()
            meta.detect_from_dataframe(df)
            self.model = CTGANSynthesizer(
                metadata=meta,
                epochs=self.epochs,
                cuda=(self.use_cuda and torch.cuda.is_available()),
            )
            self.model.fit(df)
            return self
        except Exception as e_sdv:
            raise ImportError("Install 'ctgan' or 'sdv' to use CTGAN.") from e_sdv

    def sample(self, n: int):
        if self.backend == "ctgan":
            df_syn = self.model.sample(n)
        else:
            df_syn = self.model.sample(num_rows=n)
        Zs = df_syn[self.cols].values.astype(np.float32)
        Zs = np.nan_to_num(Zs, nan=0.0, posinf=1.0, neginf=0.0)
        Zs = np.clip(Zs, 0.0, 1.0)
        return Zs[:, :self.d_x], Zs[:, self.d_x:]

## 7. 평가 지표 (KS / PCD / DCR / TSTR)
- KS: feature-wise 분포 비교
- PCD: 상관행렬 차이
- DCR: synthetic→real 최근접 거리
- TSTR: synthetic으로 학습, real test에서 R² / RMSE

In [ ]:

def ks_test_featurewise(X_real: np.ndarray, X_syn: np.ndarray, feature_names=None):
    D = X_real.shape[1]
    if feature_names is None:
        feature_names = [f"f_{i:03d}" for i in range(D)]
    rows = []
    for j in range(D):
        r = X_real[:, j]
        s = X_syn[:, j]
        if np.allclose(r, r[0]) and np.allclose(s, s[0]):
            ks_stat, pval = 0.0, 1.0
        else:
            res = stats.ks_2samp(r, s, alternative="two-sided", mode="auto")
            ks_stat, pval = float(res.statistic), float(res.pvalue)
        rows.append((feature_names[j], ks_stat, pval))
    df = pd.DataFrame(rows, columns=["feature", "ks_stat", "p_value"])
    summary = {
        "ks_mean": float(df["ks_stat"].mean()),
        "ks_median": float(df["ks_stat"].median()),
        "p_pass_rate_0.05": float((df["p_value"] > 0.05).mean()),
    }
    return df, summary

def corr_matrix_np(X: np.ndarray) -> np.ndarray:
    C = np.corrcoef(X, rowvar=False)
    return np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)

def pairwise_correlation_difference(X_real: np.ndarray, X_syn: np.ndarray):
    C_r = corr_matrix_np(X_real)
    C_s = corr_matrix_np(X_syn)
    diff = C_r - C_s
    return {
        "pcd_fro": float(np.linalg.norm(diff, ord="fro")),
        "pcd_mean_abs": float(np.mean(np.abs(diff))),
        "C_real": C_r,
        "C_syn": C_s,
        "C_diff": diff,
    }

def dcr_metrics(X_real_s: np.ndarray, X_syn_s: np.ndarray, use_gpu: bool = True, chunk: int = 2048):
    if use_gpu and torch.cuda.is_available():
        Xr = torch.from_numpy(X_real_s).to(DEVICE)
        Xs = torch.from_numpy(X_syn_s).to(DEVICE)
        mins = []
        for i in range(0, Xs.shape[0], chunk):
            part = Xs[i:i+chunk]
            dist = torch.cdist(part, Xr)
            mins.append(dist.min(dim=1).values.detach().cpu())
        d_syn = torch.cat(mins).numpy()
    else:
        nn = NearestNeighbors(n_neighbors=1).fit(X_real_s)
        d_syn, _ = nn.kneighbors(X_syn_s, n_neighbors=1)
        d_syn = d_syn.reshape(-1)

    nn2 = NearestNeighbors(n_neighbors=2).fit(X_real_s)
    d_real, _ = nn2.kneighbors(X_real_s, n_neighbors=2)
    d_real = d_real[:, 1]

    def summarize(arr):
        return dict(
            mean=float(np.mean(arr)),
            median=float(np.median(arr)),
            q05=float(np.quantile(arr, 0.05)),
            q95=float(np.quantile(arr, 0.95)),
        )

    s_syn = summarize(d_syn)
    s_real = summarize(d_real)

    return {
        "dcr_syn_mean": s_syn["mean"],
        "dcr_syn_median": s_syn["median"],
        "dcr_syn_q05": s_syn["q05"],
        "dcr_syn_q95": s_syn["q95"],
        "dcr_real_mean": s_real["mean"],
        "dcr_ratio_mean": float(s_syn["mean"] / (s_real["mean"] + 1e-12)),
    }

def eval_regression(y_true: np.ndarray, y_pred: np.ndarray):
    r2s, rmses = [], []
    for j in range(y_true.shape[1]):
        r2s.append(r2_score(y_true[:, j], y_pred[:, j]))
        rmses.append(math.sqrt(mean_squared_error(y_true[:, j], y_pred[:, j])))
    return {"r2_mean": float(np.mean(r2s)), "rmse_mean": float(np.mean(rmses))}

In [ ]:

def try_make_xgb_regressor(random_state: int = 0, use_gpu: bool = True):
    try:
        import xgboost as xgb
    except Exception as e:
        print("[XGB] not installed:", e)
        return None

    params = dict(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=random_state,
        objective="reg:squarederror",
        n_jobs=0,
    )

    if use_gpu and torch.cuda.is_available():
        try:
            model = xgb.XGBRegressor(**params, tree_method="gpu_hist", predictor="gpu_predictor")
            print("[XGB] using gpu_hist")
            return model
        except Exception:
            try:
                model = xgb.XGBRegressor(**params, tree_method="hist", device="cuda")
                print("[XGB] using device=cuda")
                return model
            except Exception as e2:
                print("[XGB] GPU mode failed -> CPU:", e2)

    model = xgb.XGBRegressor(**params, tree_method="hist")
    print("[XGB] using CPU hist")
    return model

class TorchMLPRegressor:
    def __init__(self, input_dim, output_dim, hidden=(256,128), lr=2e-3, weight_decay=1e-4,
                 max_epochs=400, batch_size=256, patience=30, device=DEVICE):
        self.device = device
        layers = []
        d = input_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.05)]
            d = h
        layers += [nn.Linear(d, output_dim)]
        self.net = nn.Sequential(*layers).to(device)

        self.lr=lr; self.weight_decay=weight_decay
        self.max_epochs=max_epochs; self.batch_size=batch_size; self.patience=patience

    def fit(self, X, y):
        Xt = torch.from_numpy(X.astype(np.float32)).to(self.device)
        yt = torch.from_numpy(y.astype(np.float32)).to(self.device)

        n = Xt.shape[0]
        idx = torch.randperm(n, device=self.device)
        n_val = max(1, int(0.2*n))
        val_idx, tr_idx = idx[:n_val], idx[n_val:]
        Xtr, ytr = Xt[tr_idx], yt[tr_idx]
        Xva, yva = Xt[val_idx], yt[val_idx]

        opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        best = float("inf")
        best_state = None
        bad = 0
        bs = min(self.batch_size, len(Xtr))

        for epoch in range(1, self.max_epochs+1):
            self.net.train()
            perm = torch.randperm(len(Xtr), device=self.device)
            tr_loss = 0.0
            for i in range(0, len(Xtr), bs):
                xb = Xtr[perm[i:i+bs]]
                yb = ytr[perm[i:i+bs]]
                opt.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                    pred = self.net(xb)
                    loss = F.mse_loss(pred, yb)

                if SCALER.is_enabled():
                    SCALER.scale(loss).backward()
                    SCALER.step(opt)
                    SCALER.update()
                else:
                    loss.backward()
                    opt.step()

                tr_loss += float(loss.detach()) * len(xb)
            tr_loss /= len(Xtr)

            self.net.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                    pred_va = self.net(Xva)
                    va_loss = float(F.mse_loss(pred_va, yva).detach())

            if va_loss < best - 1e-6:
                best = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.net.state_dict().items()}
                bad = 0
            else:
                bad += 1

            if epoch % 50 == 0 or epoch == 1:
                print(f"[TorchMLP] epoch {epoch:4d} | train {tr_loss:.6f} | val {va_loss:.6f} | best {best:.6f}")

            if bad >= self.patience:
                print(f"[TorchMLP] Early stopping at epoch {epoch} (best val {best:.6f})")
                break

        if best_state is not None:
            self.net.load_state_dict(best_state)
        return self

    def predict(self, X):
        self.net.eval()
        Xt = torch.from_numpy(X.astype(np.float32)).to(self.device)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                pred = self.net(Xt).float().cpu().numpy()
        return pred

def tstr_with_xgb(Xtr, ytr, Xte, yte, random_state=0, max_targets=20):
    # many targets can be slow; optionally limit
    model0 = try_make_xgb_regressor(random_state=random_state, use_gpu=True)
    if model0 is None:
        return None

    T = ytr.shape[1]
    T_use = min(T, max_targets)
    preds = []
    for j in range(T_use):
        mj = try_make_xgb_regressor(random_state=random_state + j, use_gpu=True)
        mj.fit(Xtr, ytr[:, j])
        preds.append(mj.predict(Xte)[:, None])
    y_pred = np.concatenate(preds, axis=1)
    return eval_regression(yte[:, :T_use], y_pred)

def tstr_with_torch_mlp(Xtr, ytr, Xte, yte, random_state=0):
    seed_everything(random_state)
    mlp = TorchMLPRegressor(
        input_dim=Xtr.shape[1],
        output_dim=ytr.shape[1],
        hidden=(256,128),
        lr=2e-3,
        weight_decay=1e-4,
        max_epochs=400,
        batch_size=256,
        patience=30,
        device=DEVICE,
    )
    mlp.fit(Xtr, ytr)
    y_pred = mlp.predict(Xte)
    return eval_regression(yte, y_pred)

## 8. 메인 실행 루프: Fit → Sample → Phase1 → Phase2(TSTR)
- 합성 샘플 수 `N_SYNTH`를 늘리면 GPU 활용이 증가합니다.
- DCR은 GPU(cdist)로 계산합니다.

In [ ]:

N_SYNTH = max(5000, 20 * len(X_train_s))  # bigger -> better GPU utilization
USE_GPU_FOR_DCR = True
PRINT_TOP_KS = 10

synthesizers = [
    BootstrapSynthesizer(0),
    IndependentKDESynthesizer(bandwidth=0.05, random_state=0),
    GaussianCopulaSynthesizer(random_state=0),
    VAESynthesizer(
        latent_dim=24,
        hidden=(256, 128),
        beta=0.6,
        lr=2e-3,
        max_epochs=600,
        batch_size=256,
        patience=60,
        noise_std=0.01,
        random_state=0,
        device=DEVICE,
    ),
]

# add CTGAN if available
try:
    _ = __import__("ctgan")
    synthesizers.append(CTGANSynthesizerWrapper(epochs=300, batch_size=500, random_state=0, use_cuda=True))
except Exception:
    print("[CTGAN] not installed -> skip")

In [ ]:

# TRTR upper bound (train real scaled -> predict real scaled target)
print("=== TRTR (Upper Bound, TorchMLP) ===")
trtr = tstr_with_torch_mlp(X_train_s, y_train_s, X_test_s, y_test_s, random_state=0)
print(trtr)

print("=== TRTR (Upper Bound, XGB - optional/limited targets) ===")
trtr_xgb = tstr_with_xgb(X_train_s, y_train_s, X_test_s, y_test_s, random_state=0, max_targets=20)
print(trtr_xgb)

In [ ]:

results = []
ks_tables = {}
corr_tables = {}
dcr_tables = {}

for syn in synthesizers:
    print("\n" + "="*80)
    print("Fitting:", syn.name)
    t0 = time.time()
    syn.fit(X_train_s, y_train_s)
    fit_sec = time.time() - t0

    print("Sampling:", syn.name, "| n =", N_SYNTH)
    t1 = time.time()
    Xs_syn, ys_syn = syn.sample(N_SYNTH)
    sample_sec = time.time() - t1

    # inverse to original domain (for KS/PCD) and target domain for TSTR
    X_syn = pre.inverse_transform_X(Xs_syn)
    y_syn = pre.inverse_transform_y(ys_syn)

    # Phase 1: KS on original X scale (as-is)
    ks_df, ks_sum = ks_test_featurewise(X_train, X_syn, feature_names=x_cols)
    ks_tables[syn.name] = ks_df

    # Phase 1: PCD on log1p space (chem-tail friendly)
    pcd = pairwise_correlation_difference(np.log1p(np.clip(X_train, 0, None)),
                                          np.log1p(np.clip(X_syn,   0, None)))
    corr_tables[syn.name] = pcd

    # Phase 1: DCR on scaled space
    dcr = dcr_metrics(X_train_s, Xs_syn, use_gpu=USE_GPU_FOR_DCR)
    dcr_tables[syn.name] = dcr

    # Phase 2: TSTR (train on synthetic scaled X -> predict scaled y, evaluate on scaled y)
    print("TSTR TorchMLP (GPU)...")
    tstr_mlp = tstr_with_torch_mlp(Xs_syn, ys_syn, X_test_s, y_test_s, random_state=0)

    print("TSTR XGB (GPU if possible, limited targets)...")
    tstr_xgb = tstr_with_xgb(Xs_syn, ys_syn, X_test_s, y_test_s, random_state=0, max_targets=20)

    row = {
        "model": syn.name,
        "fit_sec": fit_sec,
        "sample_sec": sample_sec,
        **ks_sum,
        "pcd_fro": pcd["pcd_fro"],
        "pcd_mean_abs": pcd["pcd_mean_abs"],
        **dcr,
        "tstr_mlp_r2": tstr_mlp["r2_mean"],
        "tstr_mlp_rmse": tstr_mlp["rmse_mean"],
        "tstr_xgb_r2_20tgt": None if tstr_xgb is None else tstr_xgb["r2_mean"],
        "tstr_xgb_rmse_20tgt": None if tstr_xgb is None else tstr_xgb["rmse_mean"],
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values("tstr_mlp_r2", ascending=False)
results_df

## 9. 시각화: KS Top, DCR, Correlation Heatmap(Subset)

In [ ]:

def plot_top_ks(ks_df: pd.DataFrame, topk: int = 10, title: str = ""):
    dfp = ks_df.sort_values("ks_stat", ascending=False).head(topk)
    plt.figure(figsize=(7, 4))
    plt.barh(dfp["feature"][::-1], dfp["ks_stat"][::-1])
    plt.xlabel("KS Statistic")
    plt.title(title or f"Top-{topk} KS features")
    plt.tight_layout()
    plt.show()

def plot_dcr_mean(dcr_tables):
    names = list(dcr_tables.keys())
    vals = [dcr_tables[n]["dcr_syn_mean"] for n in names]
    plt.figure(figsize=(7, 3))
    plt.bar(names, vals)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Mean DCR (scaled space)")
    plt.title("DCR Mean by Model")
    plt.tight_layout()
    plt.show()

def plot_corr_heatmap(C: np.ndarray, title: str, vmax: float = 1.0):
    plt.figure(figsize=(5, 4))
    plt.imshow(C, vmin=-vmax, vmax=vmax)
    plt.colorbar()
    plt.title(title)
    plt.tight_layout()
    plt.show()

display(results_df)

for name, ks_df in ks_tables.items():
    plot_top_ks(ks_df, topk=PRINT_TOP_KS, title=f"{name} | Top KS features")

plot_dcr_mean(dcr_tables)

# corr subset: top-variance features
var = np.var(X_train, axis=0)
top_idx = np.argsort(var)[::-1][:50]
C_real = corr_tables[synthesizers[0].name]["C_real"][np.ix_(top_idx, top_idx)]
plot_corr_heatmap(C_real, "Real Corr(log1p) (top-variance subset)", vmax=1.0)

for syn in synthesizers:
    C_syn = corr_tables[syn.name]["C_syn"][np.ix_(top_idx, top_idx)]
    plot_corr_heatmap(C_syn, f"{syn.name} Corr(log1p) (subset)", vmax=1.0)

## 10. (Optional) 저자 제공 코드 재사용 예시
`init_modeling.py`에는 아래 기능이 들어있습니다:
- `generate_X(compounds, impute=True/False)`: 메타 컬럼 drop + 결측 처리
- `regressor_performance(...)`: MSE/R² 평가
- 모델/결과 pickle I/O

원하면 아래처럼 import해서 저자 방식 preprocessing을 부분적으로 재사용할 수 있습니다.

In [ ]:

import sys
sys.path.append(str(DATA_DIR))

from init_modeling import generate_X  # authors' helper

# authors-style X generation: requires 'beer_id' column included
if "beer_id" in df.columns:
    compounds = pd.concat([df["beer_id"], df.loc[:, x_cols]], axis=1)
    X_auth = generate_X(compounds, impute=True)  # returns DataFrame indexed by beer_id
    print("authors generate_X output:", X_auth.shape)
    display(X_auth.head(3))
else:
    print("beer_id column not found in joined df; skip authors generate_X demo.")